# Bài 02 — Tích vô hướng, nhân ma trận và tích Hadamard với NumPy

## Mục tiêu

Sau notebook này, bạn có thể:

- phân biệt dot product, matrix-vector product, matrix product và Hadamard product;
- dự đoán shape trước khi chạy phép nhân;
- kiểm chứng phép nhân ma trận bằng “hàng × cột” và tổ hợp tuyến tính;
- xử lý đúng vector `(n,)`, `(1, n)`, `(n, 1)`;
- dùng `@`, `*`, broadcasting và batched matrix multiplication đúng mục đích;
- xây dựng linear layer và mạng hai lớp chỉ bằng NumPy;
- phát hiện lỗi shape và chọn thứ tự nhân tiết kiệm bộ nhớ.

> **Cách học:** trước mỗi ô code, hãy ghi shape và giá trị dự đoán ra giấy. Chỉ chạy ô sau khi đã có dự đoán.

## 0. Khởi tạo

Notebook chỉ cần NumPy. Seed `42` giúp mọi dữ liệu ngẫu nhiên có thể tái lập. Hàm `show` giúp quan sát đồng thời giá trị và shape.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

def show(name, value):
    value = np.asarray(value)
    print(f"{name}: shape={value.shape}, ndim={value.ndim}")
    print(value, "\n")

print("NumPy version:", np.__version__)

NumPy version: 2.4.3


## 1. Dot product: nhân từng cặp rồi cộng

Với hai vector cùng độ dài:

$$
\mathbf{x}^T\mathbf{y}=\sum_k x_k y_k.
$$

Hãy tự tính `1*4 + 2*(-1) + 3*2` trước khi chạy.

In [2]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([4.0, -1.0, 2.0])

pairwise_products = x * y
manual_dot = pairwise_products.sum()
operator_dot = x @ y
function_dot = np.dot(x, y)

show("x * y", pairwise_products)
show("(x * y).sum()", manual_dot)
show("x @ y", operator_dot)

assert operator_dot.shape == ()
np.testing.assert_allclose([manual_dot, operator_dot, function_dot], 8.0)

x * y: shape=(3,), ndim=1
[ 4. -2.  6.] 

(x * y).sum(): shape=(), ndim=0
8.0 

x @ y: shape=(), ndim=0
8.0 



### Scalar NumPy có shape `()`

Kết quả dot product của hai vector 1D là một scalar NumPy: số chiều bằng 0 và shape là `()`. Đây không phải vector một phần tử `(1,)` hay ma trận `(1, 1)`.

## 2. Matrix product: mỗi phần tử là hàng × cột

Nếu `A.shape == (m, n)` và `B.shape == (n, p)`, thì `A @ B` có shape `(m, p)`. Hai kích thước ở giữa phải bằng nhau.

Với ví dụ dưới đây, hãy tính tay `C[0, 1]` bằng dot product giữa `A[0, :]` và `B[:, 1]`.

In [3]:
A = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])                              # (2, 3)
B = np.array([
    [1.0, 2.0],
    [0.0, 1.0],
    [-1.0, 3.0],
])                              # (3, 2)

C = A @ B                       # (2, 2)
c_01 = A[0, :] @ B[:, 1]

show("A", A)
show("B", B)
show("C = A @ B", C)
print("C[0, 1] từ hàng × cột:", c_01)

assert C.shape == (2, 2)
assert c_01 == C[0, 1] == 13.0

A: shape=(2, 3), ndim=2
[[1. 2. 3.]
 [4. 5. 6.]] 

B: shape=(3, 2), ndim=2
[[ 1.  2.]
 [ 0.  1.]
 [-1.  3.]] 

C = A @ B: shape=(2, 2), ndim=2
[[-2. 13.]
 [-2. 31.]] 

C[0, 1] từ hàng × cột: 13.0


### Tự cài matrix product bằng vòng lặp

Đoạn dưới không nhằm thay thế `@`; nó phơi bày đúng công thức. Trong code thực tế, `@` ngắn hơn và nhanh hơn nhiều.

In [4]:
def matmul_by_loops(left, right):
    if left.ndim != 2 or right.ndim != 2:
        raise ValueError("Hàm minh họa chỉ nhận hai ma trận 2D")
    if left.shape[1] != right.shape[0]:
        raise ValueError(f"Không tương thích: {left.shape} @ {right.shape}")

    result = np.zeros((left.shape[0], right.shape[1]), dtype=float)
    for i in range(left.shape[0]):
        for j in range(right.shape[1]):
            for k in range(left.shape[1]):
                result[i, j] += left[i, k] * right[k, j]
    return result

C_loop = matmul_by_loops(A, B)
np.testing.assert_allclose(C_loop, A @ B)
show("C tính bằng 3 vòng lặp", C_loop)

C tính bằng 3 vòng lặp: shape=(2, 2), ndim=2
[[-2. 13.]
 [-2. 31.]] 



## 3. Matrix-vector product qua hai góc nhìn

Với $\mathbf{A}\mathbf{x}$:

- theo hàng: mỗi output là một hàng của `A` dot với `x`;
- theo cột: output là tổ hợp tuyến tính các cột của `A`, với hệ số lấy từ `x`.

Hai cách phải cho cùng kết quả.

In [5]:
M = np.array([
    [1.0, 2.0, 0.0],
    [-1.0, 3.0, 1.0],
])                              # (2, 3)
v = np.array([2.0, -1.0, 4.0])  # (3,)

by_operator = M @ v
by_rows = np.array([row @ v for row in M])
by_columns = v[0] * M[:, 0] + v[1] * M[:, 1] + v[2] * M[:, 2]

show("M @ v", by_operator)
show("theo từng hàng", by_rows)
show("tổ hợp các cột", by_columns)

assert by_operator.shape == (2,)
np.testing.assert_allclose(by_operator, by_rows)
np.testing.assert_allclose(by_operator, by_columns)

M @ v: shape=(2,), ndim=1
[ 0. -1.] 

theo từng hàng: shape=(2,), ndim=1
[ 0. -1.] 

tổ hợp các cột: shape=(2,), ndim=1
[ 0. -1.] 



## 4. `@` khác `*`

`@` ghép thông tin dọc theo trục chung; `*` nhân từng vị trí và có thể broadcasting. Với hai ma trận vuông cùng shape, cả hai phép đều chạy — vì vậy đây là lỗi logic dễ lọt qua hơn lỗi runtime.

In [6]:
P = np.array([[1.0, 2.0], [3.0, 4.0]])
Q = np.array([[2.0, 0.0], [1.0, -1.0]])

hadamard = P * Q
matrix_product = P @ Q

show("P * Q (Hadamard)", hadamard)
show("P @ Q (matrix product)", matrix_product)

assert hadamard.shape == matrix_product.shape == (2, 2)
assert not np.array_equal(hadamard, matrix_product)

P * Q (Hadamard): shape=(2, 2), ndim=2
[[ 2.  0.]
 [ 3. -4.]] 

P @ Q (matrix product): shape=(2, 2), ndim=2
[[ 4. -2.]
 [10. -4.]] 



### Broadcasting với `*`

`features * scale` nhân mỗi cột với một hệ số vì `(3,)` được broadcast qua các hàng của `(2, 3)`. Đây vẫn là element-wise product, không phải matrix product.

In [7]:
features = np.array([[1.0, 10.0, 100.0], [2.0, 20.0, 200.0]])  # (2, 3)
scale = np.array([1.0, 0.1, 0.01])                               # (3,)
scaled_features = features * scale                               # (2, 3)

show("features * scale", scaled_features)
np.testing.assert_allclose(scaled_features, [[1, 1, 1], [2, 2, 2]])

features * scale: shape=(2, 3), ndim=2
[[1. 1. 1.]
 [2. 2. 2.]] 



## 5. Vector 1D, vector hàng, vector cột và outer product

Trong NumPy, `(n,)` không phải vector hàng hay vector cột và `.T` không đổi shape. Thêm trục rõ ràng khi hướng của vector quan trọng.

In [8]:
u = np.array([1.0, 2.0, 3.0])
row = u[None, :]      # (1, 3)
column = u[:, None]   # (3, 1)

dot_1d = u @ u
outer = column @ row
dot_2d = row @ column

for name, value in [("u", u), ("u.T", u.T), ("row", row),
                    ("column", column), ("u @ u", dot_1d),
                    ("column @ row", outer), ("row @ column", dot_2d)]:
    show(name, value)

assert u.T.shape == (3,)
assert dot_1d.shape == ()
assert outer.shape == (3, 3)
assert dot_2d.shape == (1, 1)
np.testing.assert_allclose(outer, np.outer(u, u))

u: shape=(3,), ndim=1
[1. 2. 3.] 

u.T: shape=(3,), ndim=1
[1. 2. 3.] 

row: shape=(1, 3), ndim=2
[[1. 2. 3.]] 

column: shape=(3, 1), ndim=2
[[1.]
 [2.]
 [3.]] 

u @ u: shape=(), ndim=0
14.0 

column @ row: shape=(3, 3), ndim=2
[[1. 2. 3.]
 [2. 4. 6.]
 [3. 6. 9.]] 

row @ column: shape=(1, 1), ndim=2
[[14.]] 



## 6. Các tính chất quan trọng

Phép nhân ma trận:

- **kết hợp:** `(A @ B) @ D == A @ (B @ D)`;
- **phân phối:** `A @ (B + D) == A @ B + A @ D`;
- **không giao hoán:** thường `A @ B != B @ A`;
- **đảo thứ tự khi chuyển vị:** `(A @ B).T == B.T @ A.T`.

In [9]:
R = np.array([[1.0, 2.0], [0.0, 1.0]])
S = np.array([[2.0, 0.0], [3.0, 1.0]])
T = np.array([[1.0, -1.0], [2.0, 2.0]])

np.testing.assert_allclose((R @ S) @ T, R @ (S @ T))
np.testing.assert_allclose(R @ (S + T), R @ S + R @ T)
np.testing.assert_allclose((R @ S).T, S.T @ R.T)

show("R @ S", R @ S)
show("S @ R", S @ R)
assert not np.array_equal(R @ S, S @ R)

R @ S: shape=(2, 2), ndim=2
[[8. 2.]
 [3. 1.]] 

S @ R: shape=(2, 2), ndim=2
[[2. 4.]
 [3. 7.]] 



## 7. Linear layer cho một mẫu và một batch

Quy ước ở đây: hàng là mẫu, cột là đặc trưng; `W.shape == (in_features, out_features)`.

```text
x: (d,)       W: (d, h)   b: (h,)   -> z: (h,)
X: (N, d)     W: (d, h)   b: (h,)   -> Z: (N, h)
```

Bias `(h,)` được broadcast qua mọi hàng của batch.

In [10]:
X_batch = np.array([
    [1.0, 0.0, 2.0],
    [0.0, 1.0, -1.0],
    [2.0, 1.0, 1.0],
    [-1.0, 2.0, 0.0],
])                                      # (4 samples, 3 features)
W = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, -1.0],
])                                      # (3 inputs, 2 outputs)
b = np.array([0.5, -0.5])               # (2 outputs,)

single_output = X_batch[0] @ W + b       # (2,)
batch_output = X_batch @ W + b           # (4, 2)
manual_first = np.array([
    X_batch[0] @ W[:, 0] + b[0],
    X_batch[0] @ W[:, 1] + b[1],
])

show("single output", single_output)
show("batch output", batch_output)

assert single_output.shape == (2,)
assert batch_output.shape == (4, 2)
np.testing.assert_allclose(single_output, batch_output[0])
np.testing.assert_allclose(single_output, manual_first)

single output: shape=(2,), ndim=1
[ 3.5 -2.5] 

batch output: shape=(4, 2), ndim=2
[[ 3.5 -2.5]
 [-0.5  1.5]
 [ 3.5 -0.5]
 [-0.5  1.5]] 



### Bias `(h,)` và `(1, h)`

Cả hai đều cho output `(N, h)`, nhưng `(h,)` thường gọn hơn. Bias `(N, 1)` mang nghĩa khác: một giá trị riêng cho từng mẫu và được cộng vào mọi output feature của mẫu đó.

In [11]:
b_row = b[None, :]                       # (1, 2)
with_1d_bias = X_batch @ W + b            # (4, 2)
with_row_bias = X_batch @ W + b_row       # (4, 2)

np.testing.assert_allclose(with_1d_bias, with_row_bias)
print("Shapes:", (X_batch @ W).shape, b.shape, b_row.shape, with_row_bias.shape)

Shapes: (4, 2) (2,) (1, 2) (4, 2)


## 8. Batched matrix multiplication

Với tensor, `@` nhân trên hai trục cuối và broadcast các trục phía trước. `batch_A.shape == (2, 3, 4)` và `batch_B.shape == (2, 4, 2)` nghĩa là có hai cặp phép nhân `(3, 4) @ (4, 2)`.

In [12]:
batch_A = rng.integers(-2, 3, size=(2, 3, 4))
batch_B = rng.integers(-2, 3, size=(2, 4, 2))
batch_C = batch_A @ batch_B

show("batch_A", batch_A)
show("batch_B", batch_B)
show("batch_A @ batch_B", batch_C)

assert batch_C.shape == (2, 3, 2)
np.testing.assert_array_equal(batch_C[0], batch_A[0] @ batch_B[0])
np.testing.assert_array_equal(batch_C[1], batch_A[1] @ batch_B[1])

batch_A: shape=(2, 3, 4), ndim=3
[[[-2  1  1  0]
  [ 0  2 -2  1]
  [-1 -2  0  2]]

 [[ 1  1  1  1]
  [ 0 -2  2  0]
  [ 0 -1 -2  2]]] 

batch_B: shape=(2, 4, 2), ndim=3
[[[ 1  1]
  [ 0  2]
  [ 0  0]
  [ 0 -1]]

 [[-2  0]
  [ 2 -2]
  [ 2  2]
  [-1  1]]] 

batch_A @ batch_B: shape=(2, 3, 2), ndim=3
[[[-2  0]
  [ 0  3]
  [-1 -7]]

 [[ 1  1]
  [ 0  8]
  [-8  0]]] 



## 9. Cosine similarity từ dot product

$$
\operatorname{cos}(\mathbf{x},\mathbf{y})=
\frac{\mathbf{x}^T\mathbf{y}}{\|\mathbf{x}\|_2\|\mathbf{y}\|_2}.
$$

Hàm nên từ chối vector zero vì mẫu số bằng 0. Với nhiều embedding đã chuẩn hóa về norm 1, ma trận cosine similarity đơn giản là `E_normalized @ E_normalized.T`.

In [13]:
def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        raise ValueError("Cosine similarity không xác định với vector zero")
    return (a @ b) / denominator

e1 = np.array([1.0, 1.0, 0.0])
e2 = np.array([1.0, 0.0, 1.0])

print("cos(e1, e2) =", cosine_similarity(e1, e2))
np.testing.assert_allclose(cosine_similarity(e1, e2), 0.5)
np.testing.assert_allclose(cosine_similarity(e1, e1), 1.0)

cos(e1, e2) = 0.4999999999999999


In [14]:
embeddings = np.array([
    [1.0, 0.0, 0.0],
    [1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0],
])
normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
similarity_matrix = normalized @ normalized.T

show("pairwise cosine similarity", similarity_matrix)
np.testing.assert_allclose(np.diag(similarity_matrix), 1.0)
np.testing.assert_allclose(similarity_matrix, similarity_matrix.T)

pairwise cosine similarity: shape=(3, 3), ndim=2
[[1.    0.707 0.   ]
 [0.707 1.    0.5  ]
 [0.    0.5   1.   ]] 



## 10. Tính kết hợp và chi phí ma trận trung gian

Hai cách đặt ngoặc cho cùng kết quả toán học, nhưng kích thước trung gian khác nhau:

- `(A @ B) @ x` tạo `(1000, 1000)` = 1.000.000 phần tử;
- `A @ (B @ x)` tạo `(10,)` = 10 phần tử.

Ta dùng kích thước vừa phải để kiểm chứng giá trị mà không lãng phí bộ nhớ.

In [15]:
A_chain = rng.normal(size=(100, 5))
B_chain = rng.normal(size=(5, 100))
x_chain = rng.normal(size=(100,))

expensive_intermediate = A_chain @ B_chain   # (100, 100)
cheap_intermediate = B_chain @ x_chain       # (5,)

left_grouping = expensive_intermediate @ x_chain
right_grouping = A_chain @ cheap_intermediate

print("Số phần tử trung gian:", expensive_intermediate.size, "so với", cheap_intermediate.size)
np.testing.assert_allclose(left_grouping, right_grouping, rtol=1e-12, atol=1e-12)

Số phần tử trung gian: 10000 so với 5


## 11. Đọc thông báo lỗi shape

Đừng sửa lỗi bằng cách transpose/reshape ngẫu nhiên. Trước hết hãy viết shape và ý nghĩa từng trục. Ô dưới chủ động bắt lỗi để notebook vẫn chạy tiếp.

In [16]:
bad_left = np.zeros((4, 3))
bad_right = np.zeros((4, 2))

try:
    bad_left @ bad_right
except ValueError as error:
    print("Lỗi dự kiến:", error)
    print("Giải thích: (4, 3) @ (4, 2) sai vì 3 != 4 ở hai trục giữa.")

Lỗi dự kiến: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 4 is different from 3)
Giải thích: (4, 3) @ (4, 2) sai vì 3 != 4 ở hai trục giữa.


## Bài tập

Có 7 bài tập tăng dần và một bài tổng hợp. Với mỗi bài:

1. ghi phép toán shape ra giấy;
2. bỏ dấu `#` và hoàn thành `TODO`;
3. chạy ô bài làm rồi ô kiểm tra;
4. chỉ mở đáp án sau khi đã thử.

Các ô kiểm tra sẽ nhắc biến còn thiếu thay vì làm notebook dừng.

### Bài 1 — Dot product không dùng `@`

Tính dot product của `ex1_x` và `ex1_y` bằng phép nhân element-wise và một phép gộp. Lưu kết quả vào `ex1_dot`.

In [17]:
ex1_x = np.array([2.0, -1.0, 3.0, 0.5])
ex1_y = np.array([4.0, 2.0, -2.0, 6.0])

# TODO
# ex1_dot = ...

In [18]:
if "ex1_dot" not in globals():
    print("⏳ Chưa làm Bài 1: hãy tạo ex1_dot.")
else:
    assert np.asarray(ex1_dot).shape == ()
    np.testing.assert_allclose(ex1_dot, ex1_x @ ex1_y)
    print("✅ Bài 1 đúng")

⏳ Chưa làm Bài 1: hãy tạo ex1_dot.


<details><summary>Đáp án Bài 1</summary>

```python
ex1_dot = (ex1_x * ex1_y).sum()
```
</details>

### Bài 2 — Tính một phần tử của matrix product

Không tính toàn bộ tích trước. Dùng đúng một hàng của `ex2_A` và một cột của `ex2_B` để tạo `ex2_c12`, tức phần tử hàng `1`, cột `2` theo index NumPy.

In [19]:
ex2_A = np.arange(12, dtype=float).reshape(3, 4)  # (3, 4)
ex2_B = np.arange(20, dtype=float).reshape(4, 5)  # (4, 5)

# TODO
# ex2_c12 = ...

In [20]:
if "ex2_c12" not in globals():
    print("⏳ Chưa làm Bài 2: hãy tạo ex2_c12.")
else:
    assert np.asarray(ex2_c12).shape == ()
    np.testing.assert_allclose(ex2_c12, (ex2_A @ ex2_B)[1, 2])
    print("✅ Bài 2 đúng")

⏳ Chưa làm Bài 2: hãy tạo ex2_c12.


<details><summary>Đáp án Bài 2</summary>

```python
ex2_c12 = ex2_A[1, :] @ ex2_B[:, 2]
```
</details>

### Bài 3 — Inner product và outer product

Từ `ex3_u` và `ex3_v`, tạo:

- `ex3_inner`: scalar dot product;
- `ex3_outer`: ma trận `(3, 3)` bằng matrix multiplication sau khi thêm trục, không dùng `np.outer`.

In [21]:
ex3_u = np.array([1.0, 2.0, 3.0])
ex3_v = np.array([-1.0, 0.0, 2.0])

# TODO
# ex3_inner = ...
# ex3_outer = ...

In [22]:
if "ex3_inner" not in globals() or "ex3_outer" not in globals():
    print("⏳ Chưa làm Bài 3: hãy tạo ex3_inner và ex3_outer.")
else:
    assert np.asarray(ex3_inner).shape == ()
    assert ex3_outer.shape == (3, 3)
    np.testing.assert_allclose(ex3_inner, np.dot(ex3_u, ex3_v))
    np.testing.assert_allclose(ex3_outer, np.outer(ex3_u, ex3_v))
    print("✅ Bài 3 đúng")

⏳ Chưa làm Bài 3: hãy tạo ex3_inner và ex3_outer.


<details><summary>Đáp án Bài 3</summary>

```python
ex3_inner = ex3_u @ ex3_v
ex3_outer = ex3_u[:, None] @ ex3_v[None, :]
```
</details>

### Bài 4 — Linear layer cho batch

Tính `ex4_output` cho 5 mẫu, 4 input features và 3 output features. Không dùng vòng lặp. Sau đó tính riêng output đầu tiên vào `ex4_first` bằng một hàng của `ex4_X`.

In [23]:
ex4_X = rng.normal(size=(5, 4))
ex4_W = rng.normal(size=(4, 3))
ex4_b = np.array([0.1, -0.2, 0.3])

# TODO
# ex4_output = ...
# ex4_first = ...

In [24]:
if "ex4_output" not in globals() or "ex4_first" not in globals():
    print("⏳ Chưa làm Bài 4: hãy tạo ex4_output và ex4_first.")
else:
    assert ex4_output.shape == (5, 3)
    assert ex4_first.shape == (3,)
    np.testing.assert_allclose(ex4_output[0], ex4_first)
    np.testing.assert_allclose(ex4_output, np.stack([row @ ex4_W + ex4_b for row in ex4_X]))
    print("✅ Bài 4 đúng")

⏳ Chưa làm Bài 4: hãy tạo ex4_output và ex4_first.


<details><summary>Đáp án Bài 4</summary>

```python
ex4_output = ex4_X @ ex4_W + ex4_b
ex4_first = ex4_X[0] @ ex4_W + ex4_b
```
</details>

### Bài 5 — Ma trận cosine similarity

Mỗi hàng của `ex5_embeddings` là một embedding khác zero. Tạo:

- `ex5_normalized`: chuẩn hóa từng hàng về norm 1;
- `ex5_similarity`: cosine similarity của mọi cặp, shape `(4, 4)`.

Không dùng vòng lặp.

In [25]:
ex5_embeddings = np.array([
    [1.0, 0.0, 1.0],
    [2.0, 0.0, 2.0],
    [0.0, 1.0, 0.0],
    [1.0, 1.0, 0.0],
])

# TODO
# ex5_norms = ...
# ex5_normalized = ...
# ex5_similarity = ...

In [26]:
ex5_names = ["ex5_norms", "ex5_normalized", "ex5_similarity"]
if not all(name in globals() for name in ex5_names):
    print("⏳ Chưa làm Bài 5: hãy tạo ex5_norms, ex5_normalized và ex5_similarity.")
else:
    assert ex5_norms.shape in {(4,), (4, 1)}
    assert ex5_normalized.shape == (4, 3)
    assert ex5_similarity.shape == (4, 4)
    np.testing.assert_allclose(np.linalg.norm(ex5_normalized, axis=1), 1.0)
    np.testing.assert_allclose(ex5_similarity, ex5_similarity.T)
    np.testing.assert_allclose(np.diag(ex5_similarity), 1.0)
    np.testing.assert_allclose(ex5_similarity[0, 1], 1.0)
    print("✅ Bài 5 đúng")

⏳ Chưa làm Bài 5: hãy tạo ex5_norms, ex5_normalized và ex5_similarity.


<details><summary>Đáp án Bài 5</summary>

```python
ex5_norms = np.linalg.norm(ex5_embeddings, axis=1, keepdims=True)
ex5_normalized = ex5_embeddings / ex5_norms
ex5_similarity = ex5_normalized @ ex5_normalized.T
```
</details>

### Bài 6 — Batched matrix multiplication

Tạo `ex6_result` bằng một phép `@`. Kết quả phải tương đương việc nhân từng cặp ma trận cùng batch index.

In [27]:
ex6_left = rng.normal(size=(4, 2, 3))
ex6_right = rng.normal(size=(4, 3, 5))

# TODO
# ex6_result = ...

In [28]:
if "ex6_result" not in globals():
    print("⏳ Chưa làm Bài 6: hãy tạo ex6_result.")
else:
    assert ex6_result.shape == (4, 2, 5)
    for batch_index in range(4):
        np.testing.assert_allclose(
            ex6_result[batch_index],
            ex6_left[batch_index] @ ex6_right[batch_index],
        )
    print("✅ Bài 6 đúng")

⏳ Chưa làm Bài 6: hãy tạo ex6_result.


<details><summary>Đáp án Bài 6</summary>

```python
ex6_result = ex6_left @ ex6_right
```
</details>

### Bài 7 — Chọn thứ tự nhân hiệu quả

Cho `A(200, 8)`, `B(8, 200)`, `x(200,)`. Tạo `ex7_result` mà **không** tạo ma trận trung gian `(200, 200)`. Lưu intermediate nhỏ vào `ex7_intermediate`.

In [29]:
ex7_A = rng.normal(size=(200, 8))
ex7_B = rng.normal(size=(8, 200))
ex7_x = rng.normal(size=(200,))

# TODO
# ex7_intermediate = ...
# ex7_result = ...

In [30]:
if "ex7_intermediate" not in globals() or "ex7_result" not in globals():
    print("⏳ Chưa làm Bài 7: hãy tạo ex7_intermediate và ex7_result.")
else:
    assert ex7_intermediate.shape == (8,)
    assert ex7_result.shape == (200,)
    np.testing.assert_allclose(ex7_result, (ex7_A @ ex7_B) @ ex7_x)
    print("✅ Bài 7 đúng — intermediate chỉ có", ex7_intermediate.size, "phần tử")

⏳ Chưa làm Bài 7: hãy tạo ex7_intermediate và ex7_result.


<details><summary>Đáp án Bài 7</summary>

```python
ex7_intermediate = ex7_B @ ex7_x
ex7_result = ex7_A @ ex7_intermediate
```

Nhờ tính kết hợp, kết quả bằng `(ex7_A @ ex7_B) @ ex7_x` nhưng không tạo 40.000 phần tử trung gian.
</details>

## Bài tổng hợp — Forward pass của mạng hai lớp

Ta phân loại 6 mẫu vào 3 lớp bằng mạng:

```text
X (6, 4)
  -> linear 1: X @ W1 + b1       (6, 5)
  -> ReLU: max(0, hidden_linear)  (6, 5)
  -> linear 2: hidden @ W2 + b2   (6, 3)
  -> softmax theo từng hàng        (6, 3)
```

Hãy tạo `cap_hidden_linear`, `cap_hidden`, `cap_logits`, `cap_probabilities` và `cap_predictions`. Khi tính softmax, trừ max từng hàng trước khi `exp` để ổn định số học.

In [31]:
cap_X = rng.normal(size=(6, 4))
cap_W1 = rng.normal(scale=0.5, size=(4, 5))
cap_b1 = rng.normal(scale=0.1, size=(5,))
cap_W2 = rng.normal(scale=0.5, size=(5, 3))
cap_b2 = rng.normal(scale=0.1, size=(3,))

# TODO 1: affine transform thứ nhất
# cap_hidden_linear = ...

# TODO 2: ReLU element-wise
# cap_hidden = ...

# TODO 3: affine transform thứ hai
# cap_logits = ...

# TODO 4: stable softmax theo từng hàng
# cap_shifted_logits = ...
# cap_exp_scores = ...
# cap_probabilities = ...

# TODO 5: index lớp có xác suất lớn nhất cho mỗi mẫu
# cap_predictions = ...

In [32]:
cap_names = [
    "cap_hidden_linear", "cap_hidden", "cap_logits", "cap_shifted_logits",
    "cap_exp_scores", "cap_probabilities", "cap_predictions",
]
if not all(name in globals() for name in cap_names):
    missing = [name for name in cap_names if name not in globals()]
    print("⏳ Bài tổng hợp chưa đủ. Còn thiếu:", missing)
else:
    assert cap_hidden_linear.shape == (6, 5)
    assert cap_hidden.shape == (6, 5)
    assert cap_logits.shape == (6, 3)
    assert cap_probabilities.shape == (6, 3)
    assert cap_predictions.shape == (6,)
    assert np.all(cap_hidden >= 0)
    np.testing.assert_allclose(cap_probabilities.sum(axis=1), 1.0)
    np.testing.assert_array_equal(cap_predictions, np.argmax(cap_logits, axis=1))
    print("✅ Bài tổng hợp đúng")
    print("Xác suất 2 mẫu đầu:\n", cap_probabilities[:2])
    print("Lớp dự đoán:", cap_predictions)

⏳ Bài tổng hợp chưa đủ. Còn thiếu: ['cap_hidden_linear', 'cap_hidden', 'cap_logits', 'cap_shifted_logits', 'cap_exp_scores', 'cap_probabilities', 'cap_predictions']


<details><summary>Đáp án bài tổng hợp</summary>

```python
cap_hidden_linear = cap_X @ cap_W1 + cap_b1
cap_hidden = np.maximum(0.0, cap_hidden_linear)

cap_logits = cap_hidden @ cap_W2 + cap_b2

cap_shifted_logits = cap_logits - cap_logits.max(axis=1, keepdims=True)
cap_exp_scores = np.exp(cap_shifted_logits)
cap_probabilities = cap_exp_scores / cap_exp_scores.sum(axis=1, keepdims=True)

cap_predictions = np.argmax(cap_probabilities, axis=1)
```

`cap_b1` và `cap_b2` được broadcast qua trục batch. ReLU là element-wise; hai linear layer dùng matrix product. Softmax gộp theo trục lớp (`axis=1`) và giữ trục để broadcasting phép chia.
</details>

## Tự kiểm tra nhanh

Không chạy code, hãy trả lời:

1. `(7, 4) @ (4, 2)` cho shape nào?
2. Vì sao `(7, 4) @ (7, 4)` sai nhưng `(7, 4) * (7, 4)` đúng?
3. Với `x.shape == (4,)`, shape của `x @ x`, `x[:, None] @ x[None, :]` là gì?
4. `X(32, 10) @ W(10, 5) + b(5,)` cho shape nào? Bias được lặp theo ý nghĩa nào?
5. Trong `A @ (B @ x)`, ma trận nào tác động lên `x` trước?
6. Tại sao `(A @ B).T` phải bằng `B.T @ A.T`, không phải `A.T @ B.T`?
7. Khi embedding đã có norm 1 theo từng hàng, vì sao `E @ E.T` là ma trận cosine similarity?
8. Khi nào đổi vị trí dấu ngoặc có thể tiết kiệm rất nhiều bộ nhớ?

## Tổng kết

- Dot product nhân từng cặp phần tử rồi cộng, cho scalar với hai vector 1D.
- Matrix product tuân theo `(m, n) @ (n, p) -> (m, p)`; mỗi phần tử là hàng × cột.
- `@` là phép nhân tuyến tính; `*` là element-wise và có broadcasting.
- Matrix-vector product đồng thời là các dot product theo hàng và tổ hợp tuyến tính các cột.
- Vector NumPy `(n,)` không có hướng; thêm trục khi cần `(1, n)` hoặc `(n, 1)`.
- Phép nhân ma trận kết hợp nhưng không giao hoán; chuyển vị một tích làm đảo thứ tự.
- Linear layer cho batch có dạng `X @ W + b`; bias được broadcast qua các mẫu.
- Với tensor, `@` nhân hai trục cuối và broadcast các trục batch.
- Thứ tự đặt ngoặc có thể quyết định kích thước ma trận trung gian và chi phí tính toán.

Trước khi sang Bài 03, hãy tự viết một ví dụ trong đó `A * B` và `A @ B` đều chạy nhưng cho hai ý nghĩa hoàn toàn khác nhau.